# PSE API caller

Jeden notebook do pobierania surowych danych z publicznego API PSE. Nie wykonuje obliczeń, merge'ów ani eksportu do Excela.

1. Ustaw daty i listę endpointów w sekcji **Konfiguracja**.
2. Uruchom wszystkie komórki.
3. Wyniki znajdziesz w słowniku `data`, np. `data["crb-rozl"]`.


In [ ]:
from datetime import timedelta
from typing import Iterable

import pandas as pd
import requests

BASE_URL = "https://api.raporty.pse.pl/api"


## Dostępne zbiory

Słownik zawiera endpointy i kolumny zebrane ze wszystkich wcześniejszych notebooków PSE.

In [ ]:
PSE_DATASETS = {
    "his-wlk-cal": [
        "dtime", "demand", "jgm1", "jgm", "pv", "wi"
    ],
    "poze-redoze": [
        "dtime", "period", "wi_red_network", "wi_red_balance",
        "pv_red_network", "pv_red_balance"
    ],
    "mbp-tp": [
        "dtime", "fcr_g", "fcr_d", "afrr_g", "afrr_d",
        "mfrrd_g", "mfrrd_d", "rr_g", "rr_d"
    ],
    "cmbp-tp": [
        "dtime", "fcr_g", "fcr_d", "afrr_g", "afrr_d",
        "mfrrd_g", "mfrrd_d", "rr_g", "rr_d"
    ],
    "mbu-tu": [
        "dtime", "fcr_g", "fcr_d", "afrr_g", "afrr_d",
        "mfrrd_g", "mfrrd_d"
    ],
    "cmbu-tu": [
        "dtime", "fcr_g", "fcr_d", "afrr_g", "afrr_d",
        "mfrrd_g", "mfrrd_d"
    ],
    "crb-rozl": [
        "dtime", "cen_cost", "ckoeb_cost", "ceb_pp_cost",
        "ceb_sr_cost", "ceb_sr_afrrg_cost", "ceb_sr_afrrd_cost"
    ],
    "eb-rozl": [
        "dtime", "eb_d_pp", "eb_w_pp", "eb_afrrg", "eb_afrrd"
    ],
    "zeb-rozl": [
        "dtime", "zebpp", "pzeb_afrrg_cost", "pzeb_afrrd_cost"
    ],
    "sk": ["dtime", "sk_cost"],
    "cor": ["dtime", "cor_cost"],
}


## Konfiguracja

`END_DATE` jest włącznie. Dla każdego dnia pobierane są wszystkie okresy dostawy kończące się od 00:15 do 00:00 następnego dnia.

In [ ]:
START_DATE = "2025-01-01"
END_DATE = "2025-01-31"

# Usuń z listy endpointy, których nie potrzebujesz.
ENDPOINTS_TO_DOWNLOAD = list(PSE_DATASETS)

# True = standardowa weryfikacja certyfikatu. Możesz też podać ścieżkę do pliku CA (.pem).
# Ustawienie False jest tylko rozwiązaniem awaryjnym dla zaufanej sieci.
SSL_VERIFY = True


## Caller

In [ ]:
def _parse_dates(start_date: str, end_date: str) -> tuple[pd.Timestamp, pd.Timestamp]:
    start = pd.Timestamp(start_date).normalize()
    end = pd.Timestamp(end_date).normalize()
    if end < start:
        raise ValueError("END_DATE nie może być wcześniejsza niż START_DATE.")
    return start, end


def _date_chunks(
    start: pd.Timestamp,
    end: pd.Timestamp,
    days_per_request: int,
) -> Iterable[tuple[pd.Timestamp, pd.Timestamp]]:
    if days_per_request < 1:
        raise ValueError("days_per_request musi być dodatnie.")

    chunk_start = start
    while chunk_start <= end:
        chunk_end = min(chunk_start + timedelta(days=days_per_request - 1), end)
        yield chunk_start, chunk_end
        chunk_start = chunk_end + timedelta(days=1)


def fetch_pse_endpoint(
    endpoint: str,
    start_date: str,
    end_date: str,
    *,
    fields: list[str] | None = None,
    days_per_request: int = 31,
    timeout: int = 60,
    row_limit: int = 100_000,
    verify_ssl: bool | str = True,
) -> pd.DataFrame:
    """Pobiera jeden endpoint PSE i zwraca surowe rekordy jako DataFrame."""
    start, end = _parse_dates(start_date, end_date)
    fields = fields or PSE_DATASETS.get(endpoint)
    frames = []

    for chunk_start, chunk_end in _date_chunks(start, end, days_per_request):
        # dtime w API PSE oznacza koniec okresu dostawy.
        next_midnight = chunk_end + timedelta(days=1)
        date_filter = (
            f"dtime gt '{chunk_start:%Y-%m-%d} 00:00:00' and "
            f"dtime le '{next_midnight:%Y-%m-%d} 00:00:00'"
        )
        params = {
            "$filter": date_filter,
            "$orderby": "dtime asc",
            "$first": row_limit,
        }
        if fields:
            params["$select"] = ",".join(fields)

        try:
            response = requests.get(
                f"{BASE_URL}/{endpoint}",
                params=params,
                timeout=timeout,
                verify=verify_ssl,
            )
            response.raise_for_status()
        except requests.exceptions.SSLError as exc:
            raise RuntimeError(
                "Nie udało się zweryfikować certyfikatu SSL. "
                "Ustaw SSL_VERIFY na ścieżkę do certyfikatu CA (.pem). "
                "Awaryjnie, wyłącznie w zaufanej sieci, możesz ustawić SSL_VERIFY = False."
            ) from exc
        payload = response.json()
        rows = payload.get("value")

        if not isinstance(rows, list):
            raise ValueError(f"Nieoczekiwana odpowiedź endpointu {endpoint!r}.")
        if len(rows) >= row_limit:
            raise RuntimeError(
                f"Endpoint {endpoint!r} zwrócił limit {row_limit} rekordów. "
                "Zmniejsz days_per_request, aby nie uciąć danych."
            )

        frames.append(pd.DataFrame.from_records(rows))

    if not frames:
        return pd.DataFrame(columns=fields)

    return pd.concat(frames, ignore_index=True)


def download_pse_data(
    endpoints: Iterable[str],
    start_date: str,
    end_date: str,
    *,
    days_per_request: int = 31,
    verify_ssl: bool | str = True,
) -> dict[str, pd.DataFrame]:
    """Pobiera wskazane endpointy i zwraca słownik DataFrame'ów."""
    endpoints = list(endpoints)
    unknown = sorted(set(endpoints) - set(PSE_DATASETS))
    if unknown:
        raise KeyError(f"Nieznane endpointy: {unknown}")

    result = {}
    for endpoint in endpoints:
        print(f"Pobieranie: {endpoint}")
        result[endpoint] = fetch_pse_endpoint(
            endpoint,
            start_date,
            end_date,
            fields=PSE_DATASETS[endpoint],
            days_per_request=days_per_request,
            verify_ssl=verify_ssl,
        )

    return result


## Pobranie danych

In [ ]:
data = download_pse_data(
    ENDPOINTS_TO_DOWNLOAD,
    START_DATE,
    END_DATE,
    verify_ssl=SSL_VERIFY,
)

{endpoint: frame.shape for endpoint, frame in data.items()}


### Przykład dostępu do pobranych danych

In [ ]:
data["crb-rozl"].head()
